### **Project Title: Space-Time Clustering of Mobility Patterns and Air Quality Hotspots**

Install Required Packages
and import Libraries

In [5]:
!pip install pygeohash
!pip install folium
!pip install uszipcode
!pip install geopandas
!pip install datascience



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 732.8/732.8 kB 1.2 MB/s eta 0:00:00a 0:00:01


In [ ]:
from datascience import *
import pandas as pd
import geopandas as gpd
import pygeohash as gh
import numpy as np
from shapely.geometry import Polygon
from shapely.geometry import Point
from geopandas.tools import sjoin
import matplotlib.pyplot as plt

plt.style.use('fivethirtyeight')


## **1. Data Exploration & Cleaning**

Steps:
Inspect each dataset:

*   Check dtypes format , timestamp formats, intial statistics, spatial coordinates, and column consistency.

*   Remove erroneous coordinates like (0,0).

*   Convert timestamps to a consistent format.

*   Clean and handle missing/null values.




Read CSV files NYC_AQ, NYC_pm,nyc1 and gejson file nyc_polygon

In [115]:
#Read the CSV file containing PM sensors readings and AQ file
PM_data = pd.read_csv('https://raw.githubusercontent.com/Dr-Isam-ALJAWARNEH/fds-project-space-time-clustering-for-aq/refs/heads/main/Datasets/NYC_PM.csv',index_col=False)
AQ_data = pd.read_csv('https://raw.githubusercontent.com/Dr-Isam-ALJAWARNEH/fds-project-space-time-clustering-for-aq/refs/heads/main/Datasets/NYC_AQ.csv',index_col=False)
nyc1_data = pd.read_csv('nyc1.csv')
#Read the GeoJSON file containing neighborhood boundaries into a GeoDataFrame
nyc_neighborhoods = gpd.read_file('https://raw.githubusercontent.com/Dr-Isam-ALJAWARNEH/fds-project-space-time-clustering-for-aq/refs/heads/main/Datasets/nyc_polygon.geojson')





In [116]:
print("AQ_data",AQ_data.shape)
print("PM_data",PM_data.shape)
print("nyc1_data",nyc1_data.shape)
print("nyc_neighborhoods",nyc_neighborhoods.shape)

print("----------------------AQ_data-----------------------")
print(AQ_data.describe())
print(AQ_data.info())
print("----------------------PM_data-----------------------")
print(PM_data.describe())
print(PM_data.info())
print("----------------------nyc1_data-----------------------")
print(nyc1_data.describe())
print(nyc1_data.info())

AQ_data (169999, 31)
PM_data (118765, 33)
nyc1_data (1445285, 22)
nyc_neighborhoods (310, 5)
----------------------AQ_data-----------------------
               time       latitude      longitude           bin0  \
count  1.699990e+05  169999.000000  169999.000000  169999.000000   
mean   1.634506e+09      40.826202     -73.892555      78.485926   
std    2.197971e+06       0.017215       0.019362     148.612154   
min    1.631277e+09      40.711689     -73.934052       0.000000   
25%    1.632808e+09      40.813564     -73.911232      16.000000   
50%    1.633554e+09      40.818981     -73.892303      39.000000   
75%    1.636266e+09      40.845383     -73.870804      92.000000   
max    1.639579e+09      40.904430     -73.820808    6233.000000   

                bin1           bin2           bin3           bin4  \
count  169999.000000  169999.000000  169999.000000  169999.000000   
mean        8.056747       1.885382       0.551209       0.806628   
std        22.639578       4.61622

In [197]:
#removing unnecessary data

#######AQ_data######
bin_list=['bin0','bin1','bin2','bin3','bin4','bin5','bin6','bin7','bin8','bin9','bin10','bin11','bin12','bin13','bin14','bin15','bin16','bin17','bin18','bin19','bin20','bin21','bin22','bin23']
for i in bin_list:
    if i in AQ_data.columns:
       AQ_data = AQ_data.drop(columns=[i])
        
#######PM_data######
for i in bin_list:
    if i in PM_data.columns:
       PM_data = PM_data.drop(columns=[i])
list_pm=['pm1','pm10']
for i in list_pm:
    if i in PM_data.columns:
        PM_data = PM_data.drop(columns=[i])
        
#######nyc1_data######
list_nyc1=['Store_and_fwd_flag','RateCodeID','Passenger_count','Fare_amount','MTA_tax','Tip_amount','Extra','Tolls_amount','Ehail_fee','improvement_surcharge','Total_amount','Payment_type']
for i in list_nyc1:
    if i in nyc1_data.columns:
        nyc1_data = nyc1_data.drop(columns=[i])

#remove erroneous coordinates (0,0)
#Convert the Unix epoch time column to datetime and set it as a new column

#######AQ_data######
AQ_data = \
AQ_data[(AQ_data ['latitude']!=0) & \
       (AQ_data ['longitude'] !=0)]
AQ_data ['datetime'] = pd.to_datetime(AQ_data['time'],unit='s')

print("AQ_data",AQ_data.shape)

#######PM_data######
PM_data = \
PM_data[(PM_data ['latitude']!=0) & \
       (PM_data ['longitude'] !=0)]
PM_data ['datetime'] = pd.to_datetime(PM_data['time'],unit='s')

print("PM_data",PM_data.shape)

#######nyc1_data######
nyc1_data = \
nyc1_data[(nyc1_data ['Pickup_longitude']!=0) & \
       (nyc1_data ['Pickup_latitude'] !=0)]
nyc1_data = \
nyc1_data[(nyc1_data ['Dropoff_longitude']!=0) & \
       (nyc1_data ['Dropoff_latitude'] !=0)]

nyc1_data['lpep_pickup_datetime'] = pd.to_datetime(nyc1_data['lpep_pickup_datetime'])
nyc1_data['Lpep_dropoff_datetime'] = pd.to_datetime(nyc1_data['Lpep_dropoff_datetime'])

print("nyc1_data",nyc1_data.shape)



AQ_data (169999, 10)
PM_data (118765, 10)
nyc1_data (1441239, 14)


In [199]:
print("AQ_data time range:")
print("Oldest:", AQ_data['datetime'].min())
print("Newest:", AQ_data['datetime'].max())
AQ_data.head(5)

AQ_data time range:
Oldest: 2021-09-10 12:29:09
Newest: 2021-12-15 14:35:55


,SensorID,time,latitude,longitude,temperature,humidity,pm25,datetime,geohash,time_bin
0,NYCP2_CS01A,1631277304,40.847672,-73.869316,23.7,57.3,4.508813,2021-09-10 12:35:04,dr72rh,2021-09-10 12:20:00
1,NYCP2_CS01A,1631277308,40.847668,-73.869316,23.7,57.8,5.462420,2021-09-10 12:35:08,dr72rh,2021-09-10 12:20:00
2,NYCP2_CS01A,1631277313,40.847649,-73.869362,23.7,57.8,5.154881,2021-09-10 12:35:13,dr72rh,2021-09-10 12:20:00
3,NYCP2_CS01A,1631277318,40.847649,-73.869362,23.6,57.6,4.508813,2021-09-10 12:35:18,dr72rh,2021-09-10 12:20:00
4,NYCP2_CS01A,1631277323,40.847649,-73.869362,23.6,57.5,5.539503,2021-09-10 12:35:23,dr72rh,2021-09-10 12:20:00


In [201]:
print("PM_data time range:")
print("Oldest:", PM_data['datetime'].min())
print("Newest:", PM_data['datetime'].max())
PM_data.head(5)

PM_data time range:
Oldest: 2020-01-20 23:11:00
Newest: 2020-02-17 08:10:00


,SensorID,time,latitude,longitude,temperature,humidity,pm25,datetime,geohash,time_bin
0,NYCP1_01A,1579618560,40.847183,-73.870087,16.3,15.2,5.91,2020-01-21 14:56:00,dr72rh,2020-01-21 14:40:00
1,NYCP1_01A,1579618560,40.847183,-73.870094,16.2,15.1,1.18,2020-01-21 14:56:00,dr72rh,2020-01-21 14:40:00
2,NYCP1_01A,1579618560,40.847179,-73.870094,16.1,15.1,0.76,2020-01-21 14:56:00,dr72rh,2020-01-21 14:40:00
3,NYCP1_01A,1579618560,40.847179,-73.870094,16.1,15.2,4.48,2020-01-21 14:56:00,dr72rh,2020-01-21 14:40:00
4,NYCP1_01A,1579618560,40.847179,-73.870094,16.0,15.2,5.77,2020-01-21 14:56:00,dr72rh,2020-01-21 14:40:00


In [203]:
print("nyc1_data time range:")
print("Oldest pickup_datetime:", nyc1_data['lpep_pickup_datetime'].min())
print("Newest pickup_datetime:", nyc1_data['lpep_pickup_datetime'].max())
print("Oldest dropoff_datetime:", nyc1_data['Lpep_dropoff_datetime'].min())
print("Newest dropoff_datetime:", nyc1_data['Lpep_dropoff_datetime'].max())
nyc1_data.head(5)

nyc1_data time range:
Oldest pickup_datetime: 2016-01-01 00:00:00
Newest pickup_datetime: 2016-01-31 23:59:58
Oldest dropoff_datetime: 2016-01-01 00:00:00
Newest dropoff_datetime: 2016-02-01 23:17:17


,id,VendorID,lpep_pickup_datetime,Lpep_dropoff_datetime,Pickup_longitude,Pickup_latitude,Dropoff_longitude,Dropoff_latitude,Trip_distance,Trip_type,Pickup_geohash,Dropoff_geohash,Pickup_time_bin,Dropoff_time_bin
0,0,2,2016-01-01 00:29:24,2016-01-01 00:39:36,-73.928642,40.680611,-73.924278,40.698044,1.46,1.0,dr5rmt,dr5rtb,2016-01-01 00:20:00,2016-01-01 00:20:00
1,1,2,2016-01-01 00:19:39,2016-01-01 00:39:18,-73.952675,40.723175,-73.923920,40.761379,3.56,1.0,dr5rtj,dr5rvu,2016-01-01 00:00:00,2016-01-01 00:20:00
2,2,2,2016-01-01 00:19:33,2016-01-01 00:39:48,-73.971611,40.676105,-74.013161,40.646072,3.79,1.0,dr5rks,dr5r5z,2016-01-01 00:00:00,2016-01-01 00:20:00
3,3,2,2016-01-01 00:22:12,2016-01-01 00:38:32,-73.989502,40.669579,-74.000648,40.689034,3.01,1.0,dr5rk7,dr5rkp,2016-01-01 00:20:00,2016-01-01 00:20:00
4,4,2,2016-01-01 00:24:01,2016-01-01 00:39:22,-73.964729,40.682854,-73.940720,40.663013,2.55,1.0,dr5rky,dr5rm6,2016-01-01 00:20:00,2016-01-01 00:20:00


In [205]:
nyc_neighborhoods.head(5)

,neighborhood,boroughCode,borough,@id,geometry
0,Allerton,2,Bronx,http://nyc.pediacities.com/Resource/Neighborho...,"POLYGON ((-73.849 40.872, -73.846 40.87, -73.8..."
1,Alley Pond Park,4,Queens,http://nyc.pediacities.com/Resource/Neighborho...,"POLYGON ((-73.743 40.739, -73.744 40.739, -73...."
2,Arden Heights,5,Staten Island,http://nyc.pediacities.com/Resource/Neighborho...,"POLYGON ((-74.17 40.561, -74.17 40.561, -74.16..."
3,Arlington,5,Staten Island,http://nyc.pediacities.com/Resource/Neighborho...,"POLYGON ((-74.16 40.641, -74.16 40.641, -74.16..."
4,Arrochar,5,Staten Island,http://nyc.pediacities.com/Resource/Neighborho...,"POLYGON ((-74.061 40.593, -74.061 40.593, -74...."


## **2. Data Integration**

Aggregate all datasets into a common spatial grid and temporal resolution.

Spatial Aggregation:

*    Create geohash cells over NYC.
*    Spatially join GPS/mobility data and AQ data to these grid cells.

Temporal Aggregation:
*    Resample time to a fixed interval.
*    For each grid cell & time slice, calculate:

      *    Mobility: Count of trips, speed, or density.

      *    Air Quality: Mean PM2.5, humidity, and temprature.



🗺 Spatial Grid:

In [209]:
#Set configuration
geohash_precision= 6

In [211]:
#Generate Geohash for each tuple (long,lat)
AQ_data['geohash']=AQ_data.apply(lambda x: gh.encode(x.latitude,x.longitude,precision=geohash_precision),axis=1)
PM_data['geohash']=PM_data.apply(lambda x: gh.encode(x.latitude,x.longitude,precision=geohash_precision),axis=1)

In [212]:
# generate geohashes for pickup and dropoff points for nyc1.csv file

nyc1_data['Pickup_geohash'] = nyc1_data.apply(lambda x: gh.encode(x['Pickup_latitude'], x['Pickup_longitude'], precision=geohash_precision), axis=1)
nyc1_data['Dropoff_geohash'] = nyc1_data.apply(lambda x: gh.encode(x['Dropoff_latitude'], x['Dropoff_longitude'], precision=geohash_precision), axis=1)


⏱ Temporal Resolution:
* Round timestamps to 20min for alignment. ( we need to explain why we chhose this binning intervals)

🧮 Aggregation:
For each spatial cell (geohash) and time interval (20min):
* Count number of trips, using the total pickup and dropoff points for each (geohash, time_bin).
* Calculate mean PM2.5 , humidity, and temperature.

In [214]:
#######AQ_data#######
# Round the datetime to 20-minute intervals
AQ_data['time_bin'] = AQ_data['datetime'].dt.floor('20min')

# Group by geohash and the time_bin column
AQ_aggregated = AQ_data.groupby(['geohash', 'time_bin']).agg({
    'temperature': 'mean',
    'humidity': 'mean',
    'pm25': 'mean',
    # Add any other columns if you'd like to aggregate
}).reset_index()
print("AQ_aggregated",AQ_aggregated.shape)
AQ_aggregated.head(5)


AQ_aggregated (4585, 5)


,geohash,time_bin,temperature,humidity,pm25
0,dr5rte,2021-10-29 14:40:00,13.700000,64.000000,3.187280
1,dr5ry2,2021-10-29 13:40:00,14.000000,59.300000,3.084182
2,dr5rz9,2021-09-22 14:20:00,29.600000,60.824242,11.387078
3,dr5rz9,2021-09-22 15:00:00,29.501471,64.905147,11.063003
4,dr5rz9,2021-09-22 15:20:00,28.208036,69.866071,9.865907


In [215]:
#######PM_data#######
# Round the datetime to 20-minute intervals
PM_data ['time_bin'] = PM_data ['datetime'].dt.floor('20min')

# Group by geohash and the time_bin column
PM_aggregated = PM_data.groupby(['geohash', 'time_bin']).agg({
    'temperature': 'mean',
    'humidity': 'mean',
    'pm25': 'mean',
    # Add any other columns if you'd like to aggregate
}).reset_index()
print("PM_aggregated",PM_aggregated.shape)
PM_aggregated.head(5)


PM_aggregated (1635, 5)


,geohash,time_bin,temperature,humidity,pm25
0,dr57we,2020-02-05 13:40:00,5.40000,70.600000,0.280000
1,dr57we,2020-02-05 15:40:00,9.30000,46.100000,0.000000
2,dr5ref,2020-02-05 12:20:00,6.30000,68.681818,4.754545
3,dr5reg,2020-02-05 12:20:00,6.30000,69.102410,3.041928
4,dr5reg,2020-02-05 12:40:00,6.18961,69.945022,3.505498


In [226]:
#######nyc1_data#######

# Round the datetime to 20-minute intervals
nyc1_data['Pickup_time_bin'] = nyc1_data['lpep_pickup_datetime'].dt.floor('20min')
nyc1_data['Dropoff_time_bin'] = nyc1_data['Lpep_dropoff_datetime'].dt.floor('20min')

# Group by geohash and the time_bin column
# we will get number of trips (Pickup,Dropoff)points in each geohash during each 20-minute interval.
nyc1_Pickup_points_aggregated = nyc1_data.groupby(['Pickup_geohash', 'Pickup_time_bin']).agg({
    'VendorID': 'count',}).reset_index()
nyc1_Dropoff_points_aggregated = nyc1_data.groupby(['Dropoff_geohash', 'Dropoff_time_bin']).agg({
    'VendorID': 'count',}).reset_index()



In [228]:
print("nyc1_Pickup_points_aggregated",nyc1_Pickup_points_aggregated.shape)
nyc1_Pickup_points_aggregated.head(5)

nyc1_Pickup_points_aggregated (364720, 3)


,Pickup_geohash,Pickup_time_bin,VendorID
0,9qqhg5,2016-01-27 20:40:00,1
1,9qqj6g,2016-01-27 17:20:00,1
2,9qqj74,2016-01-29 01:00:00,1
3,9qqj74,2016-01-29 01:40:00,1
4,9qqj74,2016-01-29 23:00:00,1


In [230]:
print("nyc1_Dropoff_points_aggregated",nyc1_Dropoff_points_aggregated.shape)
nyc1_Dropoff_points_aggregated.head(5)

nyc1_Dropoff_points_aggregated (598410, 3)


,Dropoff_geohash,Dropoff_time_bin,VendorID
0,9qqhg5,2016-01-27 20:20:00,1
1,9qqj74,2016-01-28 21:00:00,1
2,9qqj74,2016-01-29 00:40:00,1
3,9qqj74,2016-01-30 18:20:00,1
4,9qqj74,2016-01-31 23:40:00,1


A clear summary of total mobility activity per (geohash area and 20-minute time bin).

In [242]:
# Rename columns for consistency
nyc1_Pickup_points_aggregated = nyc1_Pickup_points_aggregated.rename(columns={
    'Pickup_geohash': 'geohash',
    'Pickup_time_bin': 'time_bin',
    'VendorID': 'pickup_trip_count'
})

nyc1_Dropoff_points_aggregated = nyc1_Dropoff_points_aggregated.rename(columns={
    'Dropoff_geohash': 'geohash',
    'Dropoff_time_bin': 'time_bin',
    'VendorID': 'dropoff_trip_count'
})

# Merge on geohash and time_bin
nyc1_joined = pd.merge(
    nyc1_Pickup_points_aggregated,
    nyc1_Dropoff_points_aggregated,
    on=['geohash', 'time_bin'],
    how='outer'
)

# Fill missing values with 0
nyc1_joined[['pickup_trip_count', 'dropoff_trip_count']] = nyc1_joined[['pickup_trip_count', 'dropoff_trip_count']].fillna(0).astype(int)

# Add total_trip_count column
nyc1_joined['total_trip_count'] = nyc1_joined['pickup_trip_count'] + nyc1_joined['dropoff_trip_count']


nyc1_joined.head(5) #we need to study the time bin

,geohash,time_bin,pickup_trip_count,dropoff_trip_count,total_trip_count
0,9qqhg5,2016-01-27 20:20:00,0,1,1
1,9qqhg5,2016-01-27 20:40:00,1,0,1
2,9qqj6g,2016-01-27 17:20:00,1,0,1
3,9qqj74,2016-01-28 21:00:00,0,1,1
4,9qqj74,2016-01-29 00:40:00,0,1,1


Note: pd.merge(..., how='outer') — Full Outer Join
🔍 What it does:
Keeps all rows from both tables.

Fills in NaN for missing values where there is no match on the join keys.

🧠 Think of it as:
“Keep everything — even if it only exists in one side.”

Now we will join the three datasets:

nyc1_joined (trip data)
AQ_aggregated (air quality from AQ_data)
PM_aggregated (air quality from PM_data)

by merge them on ['geohash', 'time_bin'].






In [246]:
# Step 1: Merge trip data with AQ data
merged_1 = pd.merge(
    nyc1_joined,
    AQ_aggregated,
    on=['geohash', 'time_bin'],
    how='outer',  # Keep all spatial-temporal combinations
    suffixes=('', '_aq'))

# Step 2: Merge the result with PM data
final_merged = pd.merge(
    merged_1,
    PM_aggregated,
    on=['geohash', 'time_bin'],
    how='outer',
    suffixes=('', '_pm'))

# Step 3: Fill missing counts with 0, and optionally drop/rename columns if needed

final_merged[['pickup_trip_count', 'dropoff_trip_count', 'total_trip_count']] = final_merged[[
    'pickup_trip_count', 'dropoff_trip_count', 'total_trip_count']].fillna(0).astype(int)

# View result
final_merged.head(5)


,geohash,time_bin,pickup_trip_count,dropoff_trip_count,total_trip_count,temperature,humidity,pm25,temperature_pm,humidity_pm,pm25_pm
0,9qqhg5,2016-01-27 20:20:00,0,1,1,NaN,NaN,NaN,NaN,NaN,NaN
1,9qqhg5,2016-01-27 20:40:00,1,0,1,NaN,NaN,NaN,NaN,NaN,NaN
2,9qqj6g,2016-01-27 17:20:00,1,0,1,NaN,NaN,NaN,NaN,NaN,NaN
3,9qqj74,2016-01-28 21:00:00,0,1,1,NaN,NaN,NaN,NaN,NaN,NaN
4,9qqj74,2016-01-29 00:40:00,0,1,1,NaN,NaN,NaN,NaN,NaN,NaN


In [248]:
#find mean for the AQ parameters (temprature, humidity,and pm25)

# Create new columns that store the average of AQ and PM values, handling NaNs safely
final_merged['pm25_mean'] = final_merged[['pm25', 'pm25_pm']].mean(axis=1, skipna=True)
final_merged['humidity_mean'] = final_merged[['humidity', 'humidity_pm']].mean(axis=1, skipna=True)
final_merged['temperature_mean'] = final_merged[['temperature', 'temperature_pm']].mean(axis=1, skipna=True)

final_merged = final_merged.drop(columns=[
    'pm25', 'pm25_pm',
    'humidity', 'humidity_pm',
    'temperature', 'temperature_pm'])

# View result
print(type(final_merged))
final_merged.head()


<class 'pandas.core.frame.DataFrame'>


,geohash,time_bin,pickup_trip_count,dropoff_trip_count,total_trip_count,pm25_mean,humidity_mean,temperature_mean
0,9qqhg5,2016-01-27 20:20:00,0,1,1,NaN,NaN,NaN
1,9qqhg5,2016-01-27 20:40:00,1,0,1,NaN,NaN,NaN
2,9qqj6g,2016-01-27 17:20:00,1,0,1,NaN,NaN,NaN
3,9qqj74,2016-01-28 21:00:00,0,1,1,NaN,NaN,NaN
4,9qqj74,2016-01-29 00:40:00,0,1,1,NaN,NaN,NaN


🗺 Spatial join:
Performing sjoin by taking geohash center point and reflected to the NYC administrative polygons
Using point in polygon spatial join


In [257]:

# Step 1: Decode geohash center to lat/lon
final_merged[['latitude', 'longitude']] = final_merged['geohash'].apply(
    lambda g: pd.Series(gh.decode(g)))

# Step 2: Convert to GeoDataFrame with point geometry
final_merged_gdf = gpd.GeoDataFrame(final_merged,geometry=gpd.points_from_xy(final_merged['longitude'], final_merged['latitude']),
    crs="EPSG:4326" ) 

# Step 3: Ensure both GeoDataFrames are in the same CRS
nyc_neighborhoods = nyc_neighborhoods.to_crs("EPSG:4326")

# Step 4: Spatial join: match each geohash point with its neighborhood polygon
joined = gpd.sjoin(final_merged_gdf, nyc_neighborhoods, how="left", predicate="within")

joined.head(5)


,geohash,time_bin,pickup_trip_count,dropoff_trip_count,total_trip_count,pm25_mean,humidity_mean,temperature_mean,geometry,latitude,longitude,index_right,neighborhood,boroughCode,borough,@id
0,9qqhg5,2016-01-27 20:20:00,0,1,1,NaN,NaN,NaN,POINT (-115.18 36.01),36.010437,-115.175171,NaN,NaN,NaN,NaN,NaN
1,9qqhg5,2016-01-27 20:40:00,1,0,1,NaN,NaN,NaN,POINT (-115.18 36.01),36.010437,-115.175171,NaN,NaN,NaN,NaN,NaN
2,9qqj6g,2016-01-27 17:20:00,1,0,1,NaN,NaN,NaN,POINT (-115.19 36.098),36.098328,-115.186157,NaN,NaN,NaN,NaN,NaN
3,9qqj74,2016-01-28 21:00:00,0,1,1,NaN,NaN,NaN,POINT (-115.18 36.093),36.092834,-115.175171,NaN,NaN,NaN,NaN,NaN
4,9qqj74,2016-01-29 00:40:00,0,1,1,NaN,NaN,NaN,POINT (-115.18 36.093),36.092834,-115.175171,NaN,NaN,NaN,NaN,NaN


🗺 Spatial join:

Useing geohash_to_polygon new defined function to convert each geohash into a bounding box  (polygon).

Performs a polygon-in-polygon spatial join.


In [250]:
#not sure about this satage

# geohash to polygon function
def geohash_to_polygon(gh_str):
    lat, lon = gh.decode(gh_str)
    lat_err, lon_err = gh.decode_exactly(gh_str)[2:]
    lat_min = lat - lat_err
    lat_max = lat + lat_err
    lon_min = lon - lon_err
    lon_max = lon + lon_err
    return Polygon([
        (lon_min, lat_min),
        (lon_max, lat_min),
        (lon_max, lat_max),
        (lon_min, lat_max),
        (lon_min, lat_min)])


# 1. Convert geohash to polygon and then to Geodataframe
final_merged['geometry'] = final_merged['geohash'].apply(geohash_to_polygon)
print(type(final_merged))
gdf = gpd.GeoDataFrame(final_merged, geometry='geometry', crs='EPSG:4326')
print(type(gdf))


# 2. Perform polygon-to-polygon overlay to compute intersections
overlap = gpd.overlay(gdf, nyc_neighborhoods, how='intersection')

# 3. Add area of intersection
overlap['intersect_area'] = overlap.geometry.area

# Area-weighted averaging of trip counts and Uniform (unweighted) averaging of pm25, temperature, and humidity.


# 4. Ensure trip columns are numeric and fill missing with 0
for col in ['pickup_trip_count', 'dropoff_trip_count', 'total_trip_count']:
    overlap[col] = overlap[col].fillna(0)

# 5. Multiply trip counts by intersection area
overlap['weighted_pickup'] = overlap['pickup_trip_count'] * overlap['intersect_area']
overlap['weighted_dropoff'] = overlap['dropoff_trip_count'] * overlap['intersect_area']
overlap['weighted_total'] = overlap['total_trip_count'] * overlap['intersect_area']

# 6. Group by neighborhood AND time_bin
neighborhood_time_summary = overlap.groupby(['neighborhood', 'time_bin']).agg({
    'intersect_area': 'sum',
    'weighted_pickup': 'sum',
    'weighted_dropoff': 'sum',
    'weighted_total': 'sum',
    'pm25_mean': 'mean',            # Uniform average for air quality
    'temperature_mean': 'mean',
    'humidity_mean': 'mean'
}).reset_index()

# 7. Compute area-weighted trip metrics
neighborhood_time_summary['pickup_trip_aw'] = neighborhood_time_summary['weighted_pickup'] / neighborhood_time_summary['intersect_area']
neighborhood_time_summary['dropoff_trip_aw'] = neighborhood_time_summary['weighted_dropoff'] / neighborhood_time_summary['intersect_area']
neighborhood_time_summary['total_trip_aw'] = neighborhood_time_summary['weighted_total'] / neighborhood_time_summary['intersect_area']

# 8. drop intermediate weighted columns
neighborhood_time_summary = neighborhood_time_summary.drop(columns=[
    'weighted_pickup', 'weighted_dropoff', 'weighted_total', 'intersect_area'
])

# Preview the result
print(neighborhood_time_summary.head())




print(largest_only.shape)
print(overlap.shape)

<class 'pandas.core.frame.DataFrame'>
<class 'geopandas.geodataframe.GeoDataFrame'>


/var/folders/h2/2gfk43_97tqgvdv7bgtkdx9r0000gn/T/ipykernel_46291/2605011594.py:30: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  overlap['intersect_area'] = overlap.geometry.area


  neighborhood            time_bin  pm25_mean  temperature_mean  \
0     Allerton 2016-01-01 00:00:00        NaN               NaN   
1     Allerton 2016-01-01 00:20:00        NaN               NaN   
2     Allerton 2016-01-01 00:40:00        NaN               NaN   
3     Allerton 2016-01-01 01:00:00        NaN               NaN   
4     Allerton 2016-01-01 01:20:00        NaN               NaN   

   humidity_mean  pickup_trip_aw  dropoff_trip_aw  total_trip_aw  
0            NaN        0.815875         0.403274       1.219148  
1            NaN        1.000000         0.000000       1.000000  
2            NaN        0.213599         1.527841       1.741440  
3            NaN        0.887131         0.885416       1.772547  
4            NaN        1.000000         1.434249       2.434249  


NameError: name 'largest_only' is not defined